# Integrated Silver Layer - Regional Operations Base

## Why Build Integrated Silver Datasets?

**The Problem with Separate Domain Tables**:
- Analysts repeat the same joins across multiple queries
- Inconsistent join logic leads to different results
- Performance overhead from repeated join operations
- Complex queries require deep schema knowledge
- Risk of incorrect joins or missing context
---

## This Notebook: `silver_regional_operations_base`

**Purpose**: Create a comprehensive operational dataset combining grid events with asset and regional context

**Integrates**:
- `silver_grid_events` (fact table) - Outages, failures, maintenance
- `silver_asset_reference` (dimension) - Substations, capacity, age, voltage
- Region context from asset reference - Population, area, country

**Output**: `vattenfall_dev.refined.silver_regional_operations_base`

**Use Cases Enabled**:
1. **Operational Analytics**: Which asset types/ages have most outages?
2. **Regional Performance**: Outage patterns by country/region/population density
3. **Capacity Analysis**: Events relative to substation capacity
4. **Impact Assessment**: Customer impact by asset characteristics
5. **Maintenance Planning**: Age-based reliability trends
6. **Executive Dashboards**: Pre-aggregated KPIs ready to visualize

**Key Enrichments**:
- Event severity scoring with asset context
- Customer impact per MVA capacity
- Regional operational metrics
- Asset utilization during events
- Time-based patterns with regional timezone context

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

print("✓ Imports complete")

In [0]:
# Load the separate silver tables
df_events = spark.table("vattenfall_dev.refined.silver_grid_events")
df_assets = spark.table("vattenfall_dev.refined.silver_asset_reference")

print("Silver Domain Tables Loaded:")
print(f"\n1. Grid Events (Fact Table):")
print(f"   Records: {df_events.count():,}")
print(f"   Columns: {len(df_events.columns)}")
print(f"   Date Range: {df_events.agg(F.min('event_date').alias('min'), F.max('event_date').alias('max')).first()}")

print(f"\n2. Asset Reference (Dimension):")
print(f"   Records: {df_assets.count():,}")
print(f"   Columns: {len(df_assets.columns)}")

print("\n3. Join Key Verification:")
print(f"   Events join key: substation_id")
print(f"   Assets join key: substation_id")

# Check for orphaned events (events without matching assets)
orphaned = df_events.join(df_assets, "substation_id", "left_anti").count()
if orphaned > 0:
    print(f"   ⚠️  Warning: {orphaned} events without matching assets")
else:
    print(f"   ✓ All events have matching assets")

# Show sample of each
print("\nSample Grid Event:")
display(df_events.limit(1))

print("\nSample Asset Reference:")
display(df_assets.limit(1))

In [0]:
# Join events with asset reference to create integrated dataset
# Using inner join since we want only events with valid asset context
# Handle duplicate column names by renaming after the join

# Identify duplicate column names (excluding join key)
event_cols = set(df_events.columns)
asset_cols = set(df_assets.columns)
duplicate_cols = event_cols.intersection(asset_cols) - {"substation_id"}  # exclude join key

print(f"Duplicate columns to handle: {duplicate_cols}")

# Perform the join first
df_joined = df_events.join(
    df_assets,
    on="substation_id",
    how="inner"
)

# Now drop the duplicate columns from the right side (asset table)
# Keep the event version of these columns
columns_to_keep = []
for c in df_joined.columns:
    # Skip columns that would be duplicates from the asset side
    # We identify them because Spark adds a suffix or we know they're from assets
    columns_to_keep.append(c)

# Actually, let's take a different approach: select specific columns we want
# All event columns + specific asset columns (not the duplicates)
event_col_list = df_events.columns
asset_specific_cols = [c for c in df_assets.columns if c not in duplicate_cols or c == "substation_id"]

# Build the select list
select_cols = []
# Add all event columns with "events." prefix
for c in event_col_list:
    select_cols.append(f"events.{c}")
# Add non-duplicate asset columns
for c in asset_specific_cols:
    if c != "substation_id":  # Already included from events
        select_cols.append(f"assets.{c}")

df_integrated = df_events.alias("events").join(
    df_assets.alias("assets"),
    on="substation_id",
    how="inner"
).select(*select_cols)

print("Integration Join Complete:")
print(f"\n  Input Records:")
print(f"    Events:      {df_events.count():>8,}")
print(f"    Assets:      {df_assets.count():>8,}")
print(f"\n  Output Records:")
print(f"    Integrated:  {df_integrated.count():>8,}")
print(f"\n  Total Columns: {len(df_integrated.columns)}")

print("\n  Column Sources:")
events_only = len(event_col_list)
assets_added = len(asset_specific_cols) - 1  # -1 for substation_id
print(f"    From events:     {events_only} columns")
print(f"    From assets:     {assets_added} columns")
print(f"    Join key:        1 column (substation_id)")

print("\nSample Integrated Record:")
display(df_integrated.limit(1))

In [0]:
# Add enrichments that combine event and asset context

df_enriched = df_integrated

# 1. Customer Impact Intensity: customers affected per MVA capacity
df_enriched = df_enriched.withColumn(
    "impact_intensity",
    F.round(F.col("affected_customers") / F.col("capacity_mva"), 2)
)

# 2. Duration Severity Score (combine duration with event severity)
df_enriched = df_enriched.withColumn(
    "duration_severity_score",
    F.round(
        F.col("duration_minutes") * 
        F.when(F.col("severity") == "critical", 3)
         .when(F.col("severity") == "high", 2)
         .when(F.col("severity") == "medium", 1.5)
         .otherwise(1),
        2
    )
)

# 3. Asset Risk Flag: aging assets with critical events
df_enriched = df_enriched.withColumn(
    "high_risk_event",
    F.when(
        (F.col("asset_age_category").isin("aging", "mature")) & 
        (F.col("severity") == "critical"),
        True
    ).otherwise(False)
)

# 4. Regional Event Density: events per 1000 km²
df_enriched = df_enriched.withColumn(
    "regional_event_density",
    F.round(1000.0 / F.col("region_area_km2"), 4)
)

# 5. Population Impact Rate: affected customers per 100k population
df_enriched = df_enriched.withColumn(
    "population_impact_rate",
    F.round((F.col("affected_customers") / F.col("region_population")) * 100000, 2)
)

# 6. Capacity Utilization Flag during event
df_enriched = df_enriched.withColumn(
    "high_capacity_asset",
    F.col("capacity_category").isin("very_large", "large")
)

print("Business Enrichments Added:")
print("\n  ✓ impact_intensity - Customers per MVA")
print("  ✓ duration_severity_score - Weighted event severity")
print("  ✓ high_risk_event - Aging assets with critical failures")
print("  ✓ regional_event_density - Events per 1000 km²")
print("  ✓ population_impact_rate - Affected customers per 100k population")
print("  ✓ high_capacity_asset - Flag for large substations")

print(f"\nTotal Columns: {len(df_enriched.columns)}")

print("\nSample Enriched Record:")
display(df_enriched.select(
    "event_id", "substation_id", "event_type", "severity",
    "impact_intensity", "duration_severity_score", "high_risk_event",
    "population_impact_rate"
).limit(5))

In [0]:
# Validate the integrated dataset quality

print("=== Data Quality Validation ===")

# 1. Check for null values in critical fields
print("\n1. Null Values in Join Keys:")
null_checks = df_enriched.select(
    F.sum(F.when(F.col("event_id").isNull(), 1).otherwise(0)).alias("null_event_id"),
    F.sum(F.when(F.col("substation_id").isNull(), 1).otherwise(0)).alias("null_substation_id"),
    F.sum(F.when(F.col("asset_key").isNull(), 1).otherwise(0)).alias("null_asset_key"),
    F.sum(F.when(F.col("region_key").isNull(), 1).otherwise(0)).alias("null_region_key")
)
display(null_checks)

# 2. Validate enrichment calculations
print("\n2. Enrichment Value Ranges:")
enrich_stats = df_enriched.select(
    F.min("impact_intensity").alias("min_impact_intensity"),
    F.max("impact_intensity").alias("max_impact_intensity"),
    F.avg("impact_intensity").alias("avg_impact_intensity"),
    F.min("duration_severity_score").alias("min_duration_severity"),
    F.max("duration_severity_score").alias("max_duration_severity"),
    F.sum(F.when(F.col("high_risk_event") == True, 1).otherwise(0)).alias("high_risk_events_count")
)
display(enrich_stats)

# 3. Regional coverage
print("\n3. Regional Coverage:")
regional_coverage = df_enriched.groupBy("country_code", "country_name").agg(
    F.count("*").alias("event_count"),
    F.countDistinct("substation_id").alias("unique_substations"),
    F.sum("affected_customers").alias("total_customers_affected")
).orderBy(F.desc("event_count"))
display(regional_coverage)

# 4. Asset age distribution in events
print("\n4. Events by Asset Age Category:")
age_dist = df_enriched.groupBy("asset_age_category").agg(
    F.count("*").alias("event_count"),
    F.avg("duration_minutes").alias("avg_duration"),
    F.sum("affected_customers").alias("total_impact")
).orderBy(
    F.when(F.col("asset_age_category") == "new", 1)
     .when(F.col("asset_age_category") == "modern", 2)
     .when(F.col("asset_age_category") == "mature", 3)
     .when(F.col("asset_age_category") == "aging", 4)
     .otherwise(5)
)
display(age_dist)

print("\n✓ Data quality validation complete")

In [0]:
# Write the integrated dataset to refined schema
table_name = "vattenfall_dev.refined.silver_regional_operations_base"

df_enriched.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"✓ Integrated silver table written: {table_name}")

# Verify
df_verify = spark.table(table_name)
print(f"\nVerification:")
print(f"  Records: {df_verify.count():,}")
print(f"  Columns: {len(df_verify.columns)}")

# Show table properties
print(f"\nTable Properties:")
spark.sql(f"DESCRIBE EXTENDED {table_name}").filter(
    F.col("col_name").isin(["Location", "Provider", "Type"])
).show(truncate=False)

## Value Demonstration: Complex Analytics Made Simple

The integrated table enables sophisticated analysis with simple queries. Here are examples that would require complex multi-table joins without integration:

In [0]:
# Query: Which aging assets in high-capacity substations have the most critical events?
# Without integration: Would need to join events → assets → regions, then filter

print("High-Risk Assets: Aging + High Capacity + Critical Events\n")

df_risk_analysis = spark.table("vattenfall_dev.refined.silver_regional_operations_base").filter(
    (F.col("high_risk_event") == True) &
    (F.col("high_capacity_asset") == True)
).groupBy(
    "substation_id", "substation_name", "country_name", 
    "capacity_mva", "asset_age_years", "asset_age_category"
).agg(
    F.count("*").alias("critical_events"),
    F.sum("affected_customers").alias("total_customers_impacted"),
    F.avg("duration_minutes").alias("avg_duration"),
    F.max("duration_severity_score").alias("max_severity_score")
).orderBy(F.desc("critical_events"))

print("Top 10 High-Risk Assets:\n")
display(df_risk_analysis.limit(10))

print(f"\n→ This single query combines event data, asset attributes, and risk calculations")
print(f"   Without integration: 3 table joins + 2 complex filters")

In [0]:
# Query: Compare operational performance across regions
# Metrics: Event frequency, customer impact, response patterns

print("Regional Operational Performance Scorecard\n")

df_regional_perf = spark.table("vattenfall_dev.refined.silver_regional_operations_base").groupBy(
    "country_code", "country_name", "region_name"
).agg(
    F.count("*").alias("total_events"),
    F.countDistinct("substation_id").alias("affected_substations"),
    F.avg("duration_minutes").alias("avg_event_duration"),
    F.sum("affected_customers").alias("total_customer_impact"),
    F.avg("population_impact_rate").alias("avg_population_impact_rate"),
    F.sum(F.when(F.col("severity") == "critical", 1).otherwise(0)).alias("critical_events"),
    F.avg("impact_intensity").alias("avg_impact_intensity")
).orderBy(F.desc("total_events"))

print("Regional Performance Metrics:\n")
display(df_regional_perf)

print(f"\n→ Pre-calculated metrics enable instant regional dashboards")
print(f"   Without integration: Multiple aggregations across joined tables")

In [0]:
# Query: Do higher capacity substations have different failure patterns?
# Analyze event patterns by capacity category and voltage level

print("Capacity & Voltage Level Reliability Analysis\n")

df_capacity_analysis = spark.table("vattenfall_dev.refined.silver_regional_operations_base").groupBy(
    "capacity_category", "voltage_level"
).agg(
    F.count("*").alias("event_count"),
    F.countDistinct("substation_id").alias("unique_assets"),
    F.avg("duration_minutes").alias("avg_duration"),
    F.avg("affected_customers").alias("avg_customers_affected"),
    F.avg("impact_intensity").alias("avg_impact_intensity"),
    F.sum(F.when(F.col("event_type") == "equipment_failure", 1).otherwise(0)).alias("equipment_failures"),
    F.sum(F.when(F.col("event_type") == "unplanned_outage", 1).otherwise(0)).alias("unplanned_outages")
).orderBy("voltage_level", "capacity_category")

print("Capacity-Voltage Reliability Matrix:\n")
display(df_capacity_analysis)

print(f"\n→ Cross-dimensional analysis ready without complex joins")
print(f"   Without integration: Nested subqueries and multiple groupings")

In [0]:
# Final summary
df_final = spark.table("vattenfall_dev.refined.silver_regional_operations_base")

print("="*70)
print("SILVER REGIONAL OPERATIONS INTEGRATION SUMMARY")
print("="*70)

print("\n📄 Integration Composition:")
print("   • Base: silver_grid_events (fact table)")
print("   • Enriched with: silver_asset_reference (dimension)")
print("   • Join key: substation_id")

print("\n📊 Dataset Profile:")
print(f"   Total Records:     {df_final.count():>8,}")
print(f"   Total Columns:     {len(df_final.columns):>8}")
print(f"   Event Types:       {df_final.select('event_type').distinct().count():>8}")
print(f"   Unique Assets:     {df_final.select('substation_id').distinct().count():>8}")
print(f"   Countries:         {df_final.select('country_code').distinct().count():>8}")
print(f"   Date Range:        {df_final.agg(F.min('event_date')).first()[0]} to {df_final.agg(F.max('event_date')).first()[0]}")

print("\n✨ Business Enrichments Added:")
print("   ✓ impact_intensity - Customer impact per MVA capacity")
print("   ✓ duration_severity_score - Weighted event severity metric")
print("   ✓ high_risk_event - Flag for critical aging asset events")
print("   ✓ regional_event_density - Geographic event concentration")
print("   ✓ population_impact_rate - Normalized regional impact")
print("   ✓ high_capacity_asset - Large substation indicator")

print("\n🎯 Value Delivered:")
print("   1. ✅ Performance: Pre-joined - eliminates repeated join overhead")
print("   2. ✅ Consistency: Single source of truth for operational analytics")
print("   3. ✅ Usability: Simple queries for complex cross-dimensional analysis")
print("   4. ✅ Quality: Validated join logic and data quality checks built-in")
print("   5. ✅ Enrichment: Business metrics calculated consistently")

print("\n🚀 Ready for Day 4 Analytics:")
print("   • Asset reliability analysis by age/capacity/voltage")
print("   • Regional operational performance dashboards")
print("   • Customer impact assessment and trending")
print("   • Risk-based maintenance prioritization")
print("   • Executive KPI reporting")

print("\n📁 Table Location:")
print("   vattenfall_dev.refined.silver_regional_operations_base")

print("\n💡 Design Pattern:")
print("   This integration follows the 'Conformed Dimension' pattern from")
print("   Kimball dimensional modeling - creating analysis-ready fact tables")
print("   with pre-joined dimensions for optimal query performance.")

print("\n🔄 Recommended: Build Similar Integrations For:")
print("   • silver_market_operations_base (prices + market zones)")
print("   • silver_weather_operations_base (weather + assets + regions)")
print("   • silver_equipment_maintenance_base (events + equipment types)")

print("\n" + "="*70)
print("✓ INTEGRATION COMPLETE - Ready for analytical consumption")
print("="*70)


## Overall Strategic Insights
Immediate Priorities:
Replace SUB105 and SUB136 in Finland (high-risk aging assets)
Investigate Copenhagen's long restoration times (228 min avg)
Capacity expansion in Turku (1,200 population impact rate is unacceptable)
## Investment Strategy:
Finland needs most attention: 58% of events, aging infrastructure
Focus on large capacity substations: When they fail, thousands affected
Modernize extra-high voltage assets: Critical events concentrate there
## Operational Improvements:
Copenhagen: Train crews or pre-position equipment for faster restoration
Turku: Add substation capacity or grid segmentation to reduce impact
Finland-wide: Accelerated replacement program for 25+ year old assets
## KPI Targets:
Reduce population impact rate below 500 per 100k across all regions
Target < 60 min average restoration time (Copenhagen needs major improvement)
Eliminate high-risk events (replace all critical aging assets)